In [1]:
import subprocess, sys
packages = "huggingface_hub bitsandbytes>=0.46.1 transformers peft accelerate trl datasets sentencepiece huggingface_hub[hf_transfer]"
subprocess.run(f"{sys.executable} -m pip install -q -U {packages}", shell=True)
print("✅ Installation forcée et terminée")


✅ Installation forcée et terminée


In [2]:
from huggingface_hub import login
import getpass
token = getpass.getpass("Token HF : ")
login(token=token)

Token HF :  ········


In [ ]:
import random
import numpy as np
import torch

SEED = 42

def fixer_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

fixer_seed()
print(f"Seed fixée à {SEED} pour reproductibilité")

In [4]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

MODEL_NAME = "Qwen/Qwen3-1.7B-Base"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True, 
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float32,   # 👈 On passe le calcul 4-bit en float32 pour la stabilité
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, 
    quantization_config=bnb_config,
    torch_dtype=torch.float32,              # 👈 Modèle chargé globalement en float32
    device_map="auto",
)

model.config.torch_dtype = torch.float32
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
model.config.use_cache = False

lora_config = LoraConfig(
    r=16, 
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05, 
    bias="none", 
    task_type="CAUSAL_LM",
)
model = get_peft_model(model, lora_config)

# 🔥 On force TOUT en float32 pur pour tuer définitivement le bug de précision
for name, param in model.named_parameters():
    if "norm" in name.lower() or param.requires_grad:
        param.data = param.data.to(torch.float32)

print("Dtype de sécurité validé :", next(model.parameters()).dtype)


Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

Dtype de sécurité validé : torch.float32


In [5]:
from datasets import load_dataset
from transformers import DataCollatorForSeq2Seq

dataset = load_dataset("UserMarrakech/chsa-triage-medical-sft")

def formater_exemple(exemple):
    contexte = exemple.get("contexte_patient") or ""
    instruction = exemple["instruction"] or ""
    reponse = exemple["reponse"] or ""
    priorite = exemple.get("priorite")
    if contexte.strip():
        prompt = f"### Cas clinique :\n{contexte}\n\n### Question :\n{instruction}\n\n### Réponse :\n"
    else:
        prompt = f"### Question :\n{instruction}\n\n### Réponse :\n"
    reponse_complete = f"[Niveau de priorité estimé : {priorite}]\n\n{reponse}" if priorite else reponse
    return {"text": prompt + reponse_complete + tokenizer.eos_token}

dataset_formate = dataset.map(formater_exemple)

def tokeniser(exemple):
    resultat = tokenizer(exemple["text"], truncation=True, max_length=1024, padding=False)
    resultat["labels"] = resultat["input_ids"].copy()
    return resultat

dataset_tokenise = dataset_formate.map(tokeniser, batched=True, remove_columns=dataset_formate["train"].column_names, num_proc=1)

data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, padding=True, label_pad_token_id=-100)
print("✅ Dataset prêt.")


README.md:   0%|          | 0.00/894 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 3.07MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/validation-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  492kB            

data/validation-00000-of-00001.parquet: downloading bytes:           |  0.00B            

...

Map:   0%|          | 0/654 [00:00<?, ? examples/s]

Map (num_proc=1):   0%|          | 0/5227 [00:00<?, ? examples/s]

Map (num_proc=1):   0%|          | 0/653 [00:00<?, ? examples/s]

Map (num_proc=1):   0%|          | 0/654 [00:00<?, ? examples/s]

✅ Dataset prêt.


In [ ]:
import os
from huggingface_hub import snapshot_download
from trl import SFTConfig, SFTTrainer

# 1. Configuration du répertoire local cible
LOCAL_CHECKPOINT_DIR = "/kaggle/working/imported_checkpoint"

print("📥 Téléchargement du checkpoint depuis Hugging Face...")
try:
    snapshot_download(
        repo_id="UserMarrakech/qwen3-triage-medical",
        local_dir=LOCAL_CHECKPOINT_DIR,
        repo_type="model"
    )
    print("✅ Téléchargement réussi.")
except Exception as e:
    print(f"❌ Erreur lors du téléchargement : {e}")

# --- DÉTECTION DYNAMIQUE DU CHEMIN DU CHECKPOINT ---
chemin_final_checkpoint = LOCAL_CHECKPOINT_DIR

# On cherche où se trouve précisément le fichier 'trainer_state.json'
for racine, dossiers, fichiers in os.walk(LOCAL_CHECKPOINT_DIR):
    if "trainer_state.json" in fichiers:
        chemin_final_checkpoint = racine
        break

print(f"📂 Chemin cible détecté pour la reprise : {chemin_final_checkpoint}")

# 2. Configuration SFTConfig
sft_config_production = SFTConfig(
    output_dir="/kaggle/working/qwen3_triage_medical_stable",
    num_train_epochs=3,                  
    per_device_train_batch_size=8,       
    gradient_accumulation_steps=2,       
    learning_rate=5e-5,                  
    max_grad_norm=0.3,                   
    optim="paged_adamw_8bit",
    lr_scheduler_type="cosine",
    warmup_steps=20,                     
    logging_steps=10,                    
    eval_strategy="no",
    save_strategy="epoch",               
    save_total_limit=1,                  
    push_to_hub=True,                    
    hub_model_id="qwen3-triage-medical", 
    hub_private_repo=True,               
    hub_strategy="checkpoint",           
    fp16=False,                          
    bf16=False,
    report_to="none",
    dataset_text_field="text",
    dataset_kwargs={"skip_prepare_dataset": True},
    seed=SEED,
    data_seed=SEED,
)

# 3. Réinitialisation du Trainer
trainer_production = SFTTrainer(
    model=model, 
    args=sft_config_production,
    train_dataset=dataset_tokenise["train"],
    data_collator=data_collator,
)

print("\n🔄 Reprise officielle de l'entraînement...")
# On passe le dossier exact trouvé par le script
trainer_production.train(resume_from_checkpoint=chemin_final_checkpoint)


In [6]:
from huggingface_hub import HfApi

api = HfApi()
REPO_MODELE = "UserMarrakech/qwen3-triage-medical"  # vérifie que c'est bien ce nom (hub_model_id de ta config)

commits = api.list_repo_commits(REPO_MODELE)
for c in commits[:5]:
    print(f"{c.commit_id[:8]} | {c.created_at} | {c.title}")

8909920c | 2026-08-16 08:32:47+00:00 | Training in progress, epoch 3, checkpoint
45d4b9a2 | 2026-08-16 08:32:45+00:00 | Training in progress, epoch 3
9a9278e0 | 2026-08-15 20:22:29+00:00 | Training in progress, epoch 2, checkpoint
72817af4 | 2026-08-15 20:22:27+00:00 | Training in progress, epoch 2
b4471343 | 2026-08-15 15:51:35+00:00 | Training in progress, epoch 1, checkpoint


In [7]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen3-1.7B-Base")
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

In [8]:
from peft import PeftModel
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
import torch

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float32, bnb_4bit_use_double_quant=True,
)

base_model_test = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen3-1.7B-Base", quantization_config=bnb_config, dtype=torch.float32, device_map="auto",
)

REPO_MODELE = "UserMarrakech/qwen3-triage-medical"
model_final = PeftModel.from_pretrained(base_model_test, REPO_MODELE)

# Test 1 : cas BPCO (référence depuis le début)
prompt_test1 = """### Cas clinique :
Un homme de 58 ans, connu pour BPCO, se présente aux urgences pour majoration de sa dyspnée depuis 2 jours. SpO2 à 85%, FR à 32/min, cyanose des extrémités.

### Question :
Quels sont les éléments de gravité chez ce patient ?

### Réponse :
"""

inputs = tokenizer(prompt_test1, return_tensors="pt").to(model_final.device)
with torch.no_grad():
    outputs = model_final.generate(**inputs, max_new_tokens=200, temperature=0.7, do_sample=True, pad_token_id=tokenizer.eos_token_id)

print("=== TEST 1 : Cas BPCO ===")
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

adapter_config.json:   0%|          | 0.00/1.15k [00:00<?, ?B/s]

adapter_model.safetensors: reconstructing file:   0%|          |  0.00B / 69.8MB            

adapter_model.safetensors: downloading bytes:           |  0.00B            

=== TEST 1 : Cas BPCO ===
### Cas clinique :
Un homme de 58 ans, connu pour BPCO, se présente aux urgences pour majoration de sa dyspnée depuis 2 jours. SpO2 à 85%, FR à 32/min, cyanose des extrémités.

### Question :
Quels sont les éléments de gravité chez ce patient ?

### Réponse :
- <PERSON> 
 - <PERSON> 
 - PFA 
 - <PERSON>


In [10]:
def simuler_triage_corrige(contexte_patient, question_triage):
    prompt = f"### Cas clinique :\n{contexte_patient}\n\n### Question :\n{question_triage}\n\n### Réponse :\n"
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    
    with torch.no_grad():
        outputs = model_final.generate(
            **inputs,
            max_new_tokens=250,
            min_new_tokens=30,      
            temperature=0.1,        
            repetition_penalty=1.2, # Bloque les boucles répétitives comme <PERSON>
            do_sample=True,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id
        )
    
    # 🚀 LE FIX : On décode explicitement la première ligne de tenseur [0] pour garantir l'obtention d'une String
    reponse_complete = tokenizer.decode(outputs[0], skip_special_tokens=True)
    
    return reponse_complete.split("### Réponse :\n")[-1]

# --- RE-TEST DU CAS BPCO ---
cas_bpco = "Un homme de 58 ans, connu pour BPCO, se présente aux urgences pour majoration de sa dyspnée depuis 2 jours. SpO2 à 85%, FR à 32/min, cyanose des extrémités."
question_bpco = "Quels sont les éléments de gravité chez ce patient ?"

print("🩺 Nouveau test corrigé (Anti-hallucination) :")
print(simuler_triage_corrige(cas_bpco, question_bpco))


🩺 Nouveau test corrigé (Anti-hallucination) :
- Dyspnée
 - Hypoxémie 
 - Hypercapnie (CO<sub>2</sub>) avec acidoties respiratoires


In [11]:
# --- RE-TEST 2 : Cas de Traumatologie / Urgence ---
cas_urgence = (
    "Jeune homme de 22 ans amené par ses amis suite à une chute de 3 mètres d'un échafaudage. "
    "Il a perdu connaissance pendant 2 minutes. Actuellement, il est confus (score de Glasgow estimé à 13), "
    "présente une déformation majeure de la cuisse droite sans saignement visible, et se plaint d'une forte céphalée. "
    "Constantes : PA 90/50 mmHg, Pouls 120/min."
)
question_urgence = "Faire l'évaluation de triage de ce patient. Quel est le niveau de priorité estimé et la conduite à tenir ?"

print("🩺 Test de Triage Spécialisé (Évaluation de la priorité) :")
print(simuler_triage_corrige(cas_urgence, question_urgence))


🩺 Test de Triage Spécialisé (Évaluation de la priorité) :
- Étude fonctionnelle du poumon
 - <PERSON> des membres inférieurs 
 - ECG en cas suspecte de syndrome cardiaque


In [12]:
# Test 2 : cas avec signal de priorité attendu (Mme Barbie, traumatisme crânien)
cas_trauma = "Mme Barbie, 72 ans, percutée par un bus. Trauma crânien, plaie au cuir chevelu. GCS = 12, TA = 90/52, FC = 95, SpO2 = 98%, FR = 21. Présente une somnolence progressive."
question_trauma = "Quel est le niveau de priorité de ce patient et pourquoi ?"

print("🩺 Test 2 - Cas trauma avec priorité :")
print(simuler_triage_corrige(cas_trauma, question_trauma))
print()

# Test 3 : cas plus bénin (différée attendue)
cas_benin = "Un homme de 35 ans consulte pour une lésion cutanée inflammatoire au niveau du cou, chaude et douloureuse, sans fièvre ni autre symptôme."
question_benin = "Quel est le niveau de priorité de ce patient ?"

print("🩺 Test 3 - Cas bénin :")
print(simuler_triage_corrige(cas_benin, question_benin))

🩺 Test 2 - Cas trauma avec priorité :
Prioritaire car il existe des risques vitaux : traumatisme cérébral (risque d'hypoxémie), hypotension artérielle

🩺 Test 3 - Cas bénin :
<PERSON> : diagnostic prématuré
- <PERSON>
 - Diagnostic évoqué (infection bactérienne) 
Diagnostic confirmateur


In [13]:
def simuler_triage_v2(contexte_patient, question_triage):
    prompt = f"""### Cas clinique :
{contexte_patient}

### Question :
{question_triage}

Réponds en commençant systématiquement par : [Niveau de priorité estimé : urgence_maximale / urgence_moderee / differee], puis justifie en une ou deux phrases.

### Réponse :
[Niveau de priorité estimé :"""

    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    with torch.no_grad():
        outputs = model_final.generate(
            **inputs, max_new_tokens=150, min_new_tokens=20,
            temperature=0.1, repetition_penalty=1.2, do_sample=True,
            pad_token_id=tokenizer.pad_token_id, eos_token_id=tokenizer.eos_token_id,
        )
    reponse_complete = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return reponse_complete.split("### Réponse :\n")[-1]

# Re-test sur les 3 cas
print("🩺 Test BPCO (prompt renforcé) :")
print(simuler_triage_v2(cas_bpco, question_bpco))
print()

print("🩺 Test trauma (prompt renforcé) :")
print(simuler_triage_v2(cas_trauma, question_trauma))
print()

print("🩺 Test bénin (prompt renforcé) :")
print(simuler_triage_v2(cas_benin, question_benin))

🩺 Test BPCO (prompt renforcé) :
[Niveau de priorité estimé : <PERSON>]
- Hypoxémie sévère
 - Hypercapnie (pH bas)
 - Signes d’insuffisance cardiaque 
   + tachycardies et troubles du rythme ventriculaire

🩺 Test trauma (prompt renforcé) :
[Niveau de priorité estimé : urgences maximales] 
- car il existe des risques vitaux (traumatisme cérébral)

🩺 Test bénin (prompt renforcé) :
[Niveau de priorité estimé : urgences maximales]
Justification: 
- L'absence totale des signes vitaux (perte conscience) exclut toute complication grave
 - La présence d'une inflammation à la surface avec douleur indiquent un risque élevé
